In [ ]:
#| default_exp cli

# CLI WebDAV Archiving Utilities

> This module provides a CLI for the Core WebDAV Archiving Utilities.


This notebook defines the `upload-dataset` command. The command is installed by `pip install -e .` from the line
`console_scripts = upload-dataset=dataset_archiver.cli:main` in `settings.ini`, and runs `main`, which parses the options
and calls `core.archive_dataset`. The usage guide is at the end of this notebook.

In [ ]:
#| export
import sys, argparse
from dataset_archiver.core import archive_dataset, AuthError, RemoteError, DEFAULT_TIMEOUT

In [ ]:
#| export
def build_cli():
    "Create CLI entry point for dataset archival"
    p = argparse.ArgumentParser(description="Archive dataset to WebDAV with metadata and verification (resumable).")
    p.add_argument("--source", required=True, help="Path to the dataset folder")
    p.add_argument("--label", required=True, help="Dataset label (e.g. 2025_exp01)")
    p.add_argument("--title", required=True, help="Dataset title")
    p.add_argument("--source_name", required=True, help="Who collected the data")
    p.add_argument("--date_collected", required=True, help="Date the data was collected")
    p.add_argument("--description", required=True, help="Short dataset description")
    p.add_argument("--responsible", required=True, help="Person responsible for the dataset")
    p.add_argument("--username", required=True, help="WebDAV username (e.g. a012197)")
    p.add_argument("--email", required=True, help="Contact email")
    p.add_argument("--webdav", default="https://u-drive.unipi.it/remote.php/webdav/", help="Base WebDAV URL")
    p.add_argument("--registry", default="upload_registry.csv", help="Local CSV registry path")
    p.add_argument("--remote_registry", default="https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/_registry/upload_registry.csv",
                   help="Shared remote registry path")
    p.add_argument("--merged_registry", default="upload_registry_merged.csv", help="Merged registry backup path")
    p.add_argument("--dry-run", action="store_true", help="List the remote folder and show the plan; send no MKCOL or PUT")
    p.add_argument("--verify", action="store_true", help="Verify uploaded files (names and sizes)")
    p.add_argument("--strict", action="store_true", help="Fail if verification finds mismatches")
    p.add_argument("--no-sync-registry", action="store_true", help="Do not sync the registry after upload")
    p.add_argument("--archive-date", default=None, help="YYYYMMDD of the original run (resume into the same folder)")
    p.add_argument("--version-hash", default=None, help="8-char hash of the original run; skips re-hashing; folder must exist")
    p.add_argument("--overwrite-mismatched", action="store_true",
                   help="Replace remote files whose size differs from the local file (default: report and leave them)")
    p.add_argument("--max-consecutive-failures", type=int, default=5, help="Stop after this many failures in a row")
    p.add_argument("--read-timeout", type=float, default=DEFAULT_TIMEOUT[1], help="Seconds per socket read/write")
    return p

def main(argv=None):
    "Entry point for `upload-dataset`. Returns 0 when complete (or a dry run), 1 when incomplete, 2 on a fatal error."
    # Print each line as soon as it is written, even when piped to tee; otherwise Python holds output back in
    # 8 KB blocks and a working run looks stuck.
    try: sys.stdout.reconfigure(line_buffering=True)
    except (AttributeError, ValueError): pass
    args = build_cli().parse_args(argv)
    metadata = {"title": args.title, "source": args.source_name, "date_collected": args.date_collected,
                "description": args.description, "responsible": args.responsible}
    try:
        summary = archive_dataset(
            source_folder=args.source, base_webdav_url=args.webdav, label=args.label, metadata=metadata,
            username=args.username, contact_email=args.email, registry_path=args.registry,
            remote_registry_url=args.remote_registry, merged_local_backup_path=args.merged_registry,
            verify=args.verify, strict=args.strict, sync_registry=not args.no_sync_registry, dry_run=args.dry_run,
            archive_date=args.archive_date, version_hash=args.version_hash,
            overwrite_mismatched=args.overwrite_mismatched, max_consecutive_failures=args.max_consecutive_failures,
            read_timeout=args.read_timeout)
    except (AuthError, RemoteError, ValueError, FileNotFoundError, NotADirectoryError) as e:
        print(f"\n❌ {e}", file=sys.stderr)
        return 2
    return 0 if (args.dry_run or summary["complete"]) else 1

In [ ]:
#| export
#| eval: false
if __name__ == "__main__" and "ipykernel" not in sys.modules:
    sys.exit(main())

## Using `upload-dataset`

Run it from the root of the repository, so that the local registry (`upload_registry.csv`) and any failure logs are
written there.

### New dataset

```bash
upload-dataset \
  --source "/full/path/to/DATASET" \
  --label 2026_DATASETNAME \
  --title "Short title" \
  --source_name "Who collected the data" \
  --date_collected "2026-05-01--2026-05-03" \
  --description "What the data is and how it was collected." \
  --responsible "Andrea Munafo" \
  --username a012197 \
  --email andrea.munafo@unipi.it \
  --webdav https://u-drive.unipi.it/remote.php/webdav \
  --dry-run
```

Then the same without `--dry-run`, with `--verify`, and for a long upload on a Mac:

```bash
caffeinate -is upload-dataset ...same arguments... --verify 2>&1 | tee "upload_DATASETNAME_$(date +%Y%m%d_%H%M).log"
```

### Resuming an interrupted upload

A run that stops for any reason (network trouble, rate limiting, Ctrl-C) can be continued by running the same command again
with the remote folder pinned. The first lines of every run print the values to use:

```
📁 Remote folder: .../archival_data/2025_test_dataset_20260927_1802d4f6
   To resume into this folder later: --archive-date 20260927 --version-hash 1802d4f6
```

```bash
upload-dataset ... --archive-date 20260927 --version-hash 1802d4f6 --dry-run   # read-only: shows what is already there
upload-dataset ... --archive-date 20260927 --version-hash 1802d4f6 --verify
```

The date and hash are also in the folder name on u-drive (`LABEL_YYYYMMDD_HASH`). Keep the same label and metadata as
the first run.

### Options

| Option | Meaning |
|---|---|
| `--source` | local dataset folder (required) |
| `--label` | dataset label, first part of the remote folder name, e.g. `2026_EXP01` (required) |
| `--title`, `--source_name`, `--date_collected`, `--description`, `--responsible` | metadata written to `metadata.json`, `README.txt` and the registry (required) |
| `--username`, `--email` | u-drive user name, and the contact address written into the licence and README (required) |
| `--webdav` | WebDAV root; default `https://u-drive.unipi.it/remote.php/webdav/` |
| `--dry-run` | log in, list and print the plan; upload nothing |
| `--verify` | after uploading, compare local and remote files by name and size |
| `--strict` | with `--verify`, fail with an error if anything differs; use it only when you expect a complete upload |
| `--archive-date YYYYMMDD` | resume: date part of the existing folder name |
| `--version-hash HASH` | resume: hash part of the existing folder name; skips re-hashing; the folder must exist |
| `--overwrite-mismatched` | replace remote files whose size differs from the local file (off by default) |
| `--max-consecutive-failures N` | stop after N failures in a row; default 5 |
| `--read-timeout SECONDS` | time allowed for each network read or write; default 300 |
| `--registry PATH` | local registry file; default `upload_registry.csv` |
| `--remote_registry URL` | shared registry on the server; default `.../archival_data/_registry/upload_registry.csv` |
| `--merged_registry PATH` | where the merged registry is saved locally; default `upload_registry_merged.csv` |
| `--no-sync-registry` | do not update the shared registry |

For tests, combine a test label with `--registry /tmp/test_registry.csv --no-sync-registry`, so neither registry is touched.

### Exit status

0 when the upload is complete, or after a dry run; 1 when it is incomplete (run the same command again to resume);
2 for a fatal error such as a wrong password, a wrong pin or a missing source folder. The command prints each line as
soon as it is written, also when piped to `tee`.

### Installing the command

The command is installed once with `pip install -e .` in the repository. With this editable install, code changes take
effect after `nbdev_export` without reinstalling; reinstall only after changing `console_scripts` in `settings.ini`.
`upload-dataset --help` lists the options of the installed version.